In [1]:
import json
import re
from bs4 import BeautifulSoup

In [2]:
# Načtení cest k profilům
html_files = {
    "PARTY_ADDRESS": "../invalid/import/Profile_PARTY_ADDRESS.html",
    "PARTY_CONTACT": "../invalid/import/Profile_PARTY_CONTACT.html",
    "PART_PARTY": "../invalid/import/Profile_PART_PARTY.html",
    "PROD_CONTRACT": "../invalid/import/Profile_PROD_CONTRACT.html"
}

In [3]:
# Parsování HTML profilovacího souboru
def parse_html_profile(filepath):
    with open(filepath, encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")
    table = soup.find_all("table")[0]
    rows = table.find_all("tr")[1:]

    result = []
    for row in rows:
        cols = row.find_all("td")
        if len(cols) < 15:
            continue

        col_name = cols[0].text.strip()
        try:
            not_nulls = int(cols[5].text.strip())
        except ValueError:
            not_nulls = 0

        try:
            uniques = int(cols[7].text.strip())
        except ValueError:
            uniques = 0

        examples = cols[13].text.strip()
        pattern_text = cols[14].text.strip()
        patterns = re.findall(r"\('(.+?)',\s*[\d.]+\)", pattern_text)

        result.append({
            "column": col_name,
            "notnulls": not_nulls,
            "uniques": uniques,
            "examples": examples,
            "patterns": patterns,
        })
    return result

In [4]:
# Aplikace pravidel na jeden sloupec
def analyze_column(table, column_info, total_rows):
    suspicious = []

    col = column_info["column"]
    notnulls = column_info["notnulls"]
    uniques = column_info["uniques"]
    patterns = column_info["patterns"]
    examples = column_info["examples"]

    # Prázdný sloupec
    if notnulls == 0:
        suspicious.append({
            "table": table,
            "column": col,
            "issue": "empty_column",
            "description": "Sloupec nemá žádné hodnoty"
        })
        return suspicious

    # Konstantní hodnota
    if uniques == 1:
        suspicious.append({
            "table": table,
            "column": col,
            "issue": "constant_value",
            "description": "Sloupec má pouze jednu unikátní hodnotu",
            "examples": examples
        })

    # Většinově chybějící data
    if notnulls / total_rows < 0.05:
        suspicious.append({
            "table": table,
            "column": col,
            "issue": "mostly_missing",
            "description": f"Sloupec má méně než 5 % vyplněných hodnot ({notnulls} z {total_rows})"
        })

    # Doménová validace CNTR_PAY_FREQ
    if col.upper() == "CNTR_PAY_FREQ":
        allowed = {"0", "1", "2", "4", "12"}  # jednorázově, měsíčně, pololetně, kvartálně, ročně
        try:
            example_dict = eval(examples)
            if isinstance(example_dict, dict):
                for key in example_dict:
                    if key.strip() not in allowed:
                        suspicious.append({
                            "table": table,
                            "column": col,
                            "issue": "invalid_payment_frequency",
                            "description": f"Podezřelá hodnota frekvence plateb: {key}",
                            "pattern": key
                        })
        except Exception:
            pass

    return suspicious

In [5]:
# Audit
audit_results = []

for table_name, file_path in html_files.items():
    print(f"Zpracovávám: {table_name}")
    column_infos = parse_html_profile(file_path)
    total_rows = max(col["notnulls"] for col in column_infos)
    for col in column_infos:
        audit_results.extend(analyze_column(table_name, col, total_rows))

Zpracovávám: PARTY_ADDRESS
Zpracovávám: PARTY_CONTACT
Zpracovávám: PART_PARTY
Zpracovávám: PROD_CONTRACT


In [6]:
# Uložení výstupu do JSON
with open("../invalid/export/audit_results.json", "w", encoding="utf-8") as f:
    json.dump(audit_results, f, indent=2, ensure_ascii=False)